# rl16_report

> Phase 16 analysis (`RL16_PLAN`): run values over the 15 main boards (`boardValues`, `runValues`), exact one-sided tests with the training run as the unit (`permTest`, `signFlip`), the hierarchical bootstrap (runs, then boards; `hierBoot`), `rl16Summaries` (H16-1 to H16-4, Holm, claim verdicts, drift, seed spread), `writeRL16`, `figRL16`, and `syntheticRows` for tests and the quick demo. All findings are exploratory.

In [ ]:
#| default_exp rl16_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Run values and exact tests

Rows are evaluation rows (`level` `<model>|main`, `seed` the board seed). Boards with no solves have `soloShare` nan and are skipped. With 4-5 runs per arm the percentile bootstrap undercovers (about 0.90 for a nominal 0.95 in simulation), so the exact tests carry the inference and the intervals are descriptive.

In [ ]:
#| export
import itertools

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.coop_grid import MAIN_SEEDS
from explRL.fix_stats import holmFamily
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _f
from explRL.rl16_grid import RL16_ALL, RL16_PLAN, RL16_REUSED, RL16_RUNS, rl16Name

METRICS = ('sharePerHeld', 'soloShare', 'welfare')
NEW = [rl16Name(a, s) for a, s in RL16_RUNS]


def _boards(rows, model, metric):
    return {int(_f(r['seed'])): _f(r[metric]) for r in rows if r['level'] == f'{model}|main'}


def boardValues(rows12, rows13, rows16, metric):
    "{run: {board seed: value}} for the nine runs and the warm start; raises unless each has the 15 main boards."
    src = dict(eval=rows12, eval_p13=rows13)
    out = {name: _boards(src[d], m, metric) for name, (d, m) in RL16_REUSED.items()}
    out.update({n: _boards(rows16, n, metric) for n in NEW})
    bad = [n for n, b in out.items() if sorted(b) != sorted(MAIN_SEEDS)]
    if bad:
        raise ValueError(f'incomplete main-board rows for {bad}: need all nine runs and the warm start, 15 boards each')
    return out


def _mean(b):
    v = np.array([x for x in (b.values() if isinstance(b, dict) else b) if x == x], float)
    return float(v.mean()) if len(v) else NAN


def runValues(rows12, rows13, rows16, metric):
    "{run: nan-aware mean over its 15 main boards}."
    return {n: _mean(b) for n, b in boardValues(rows12, rows13, rows16, metric).items()}


def permTest(a, b):
    "Exact one-sided permutation test of mean(a) - mean(b) > 0 over all relabelings (ties count)."
    a, b = [x for x in a if x == x], [x for x in b if x == x]
    pool, k = np.array(a + b, float), len(a)
    obs = float(np.mean(a) - np.mean(b))
    stats = [pool[list(i)].mean() - np.delete(pool, list(i)).mean() for i in itertools.combinations(range(len(pool)), k)]
    return dict(diff=obs, p=float(np.mean([s >= obs - 1e-12 for s in stats])))


def signFlip(x):
    "Exact one-sided sign-flip test of mean(x) > 0 over all 2^n sign patterns (ties count)."
    x = np.array([v for v in x if v == v], float)
    obs = float(x.mean())
    stats = [(x * np.array(s)).mean() for s in itertools.product((1, -1), repeat=len(x))]
    return dict(mean=obs, p=float(np.mean([s >= obs - 1e-12 for s in stats])))


def hierBoot(groups, stat, nboot=2000, seed=0):
    "95% percentile interval of stat({arm: run means}) resampling runs within each arm, then boards within each run."
    rng = np.random.default_rng(seed)
    g = {arm: [np.array([v for v in b.values() if v == v], float) for b in runs.values()] for arm, runs in groups.items()}
    out = []
    for _ in range(nboot):
        draw = {arm: np.array([runs[i][rng.integers(0, len(runs[i]), len(runs[i]))].mean()
                               for i in rng.integers(0, len(runs), len(runs))]) for arm, runs in g.items()}
        out.append(stat(draw))
    out = np.array(out, float)
    out = out[np.isfinite(out)]
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))] if len(out) else [NAN, NAN]

## Summaries

In [ ]:
#| export
def rl16Summaries(rows12, rows13, rows16, nboot=2000):
    """RL16_PLAN: per run its values and gain; H16-1 to H16-4 (exact tests, Holm over the primary three, hierarchical
    intervals); the claim verdicts; the drift count; per arm and metric the seed standard deviation."""
    bv = {m: boardValues(rows12, rows13, rows16, m) for m in METRICS}
    warm = bv['welfare']['warm']
    gainB = {n: {sd: v - warm[sd] for sd, v in bv['welfare'][n].items()} for runs in RL16_ALL.values() for n in runs}
    rv = {m: {n: _mean(b) for n, b in bv[m].items()} for m in METRICS}
    arm = lambda m, a: [rv[m][n] for n in RL16_ALL[a]]
    grp = lambda src: {a: {n: src[n] for n in runs} for a, runs in RL16_ALL.items()}
    diff = lambda hi, lo: (lambda d: d[hi].mean() - d[lo].mean())
    H = {}
    for h, m, hi, lo in (('H16_1', 'sharePerHeld', 'indpen', 'individual'), ('H16_2', 'soloShare', 'individual', 'indpen'),
                         ('H16_3', 'welfare', 'indpen', 'individual')):
        H[h] = dict(permTest(arm(m, hi), arm(m, lo)), metric=m, ci=hierBoot(grp(bv[m]), diff(hi, lo), nboot))
    holmFamily(H, RL16_PLAN['families']['primary'])
    g4 = signFlip([_mean(gainB[n]) for n in RL16_ALL['indpen']])
    H['H16_4'] = dict(g4, metric='gain', ci=hierBoot({'indpen': {n: gainB[n] for n in RL16_ALL['indpen']}},
                                                     lambda d: d['indpen'].mean(), nboot))
    a = RL16_PLAN['alpha']
    for k in RL16_PLAN['families']['primary']:
        H[k]['supported'] = bool(H[k]['pHolm'] < a)
    H['H16_4'].update(pHolm=NAN, primary=False, supported=bool(H['H16_4']['p'] < a))
    drift = sum(rv['soloShare'][n] > 0.5 for n in RL16_ALL['individual'])
    claims = dict(sharingTested=H['H16_1']['supported'] and H['H16_2']['supported'],
                  scoreContrastTested=H['H16_3']['supported'],
                  beatsImitation=H['H16_4']['supported'] and H['H16_4']['ci'][0] > 0,
                  driftReplicated=bool(drift >= 3))
    runs = {n: dict({m: rv[m][n] for m in METRICS}, gain=_mean(gainB[n]) if n in gainB else 0.0)
            for n in ['warm'] + RL16_ALL['individual'] + RL16_ALL['indpen']}
    spread = {m: dict({x: float(np.std(arm(m, x), ddof=1)) for x in RL16_ALL},
                      between=float(np.mean(arm(m, 'indpen')) - np.mean(arm(m, 'individual')))) for m in METRICS}
    return dict(runs=runs, hyp=H, claims=claims, drift=int(drift), spread=spread)

## Report, figure, synthetic rows

In [ ]:
#| export
def _g(x, d=3):
    return 'n/a' if x is None or x != x else f'{x:.{d}f}'


def writeRL16(path, S, meta):
    "report.md: provenance, hypotheses, claim verdicts, every run, the seed spread; exploratory."
    L = [f"# Phase 16: more training seeds for the incentive result ({meta['mode']})", '', 'Exploratory, pre-specified in '
         '`RL16_PLAN`; the training run is the unit (exact tests); intervals are hierarchical bootstrap (runs, then boards) '
         'and undercover with 4-5 runs per arm.', '', f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; "
         f"GPU hours of the new runs: {meta.get('gpuHours', 'n/a')}", '', '## Hypotheses', '',
         '| ID | metric | estimate [95% CI] | p (exact) | p (Holm) | supported |', '|---|---|---|---|---|---|']
    for k, v in S['hyp'].items():
        est = v.get('diff', v.get('mean'))
        L.append(f"| {k.replace('_', '-')} | {v['metric']} | {_g(est)} [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {_g(v['p'], 4)} | "
                 f"{_g(v['pHolm'], 4)} | {'yes' if v['supported'] else 'no'} |")
    c = S['claims']
    L += ['', '## Claim verdicts (rules fixed in RL16_PLAN)', '',
          f"- Sharing result tested (H16-1 and H16-2): **{'yes' if c['sharingTested'] else 'no'}**",
          f"- Score contrast against the unfixed reward tested (H16-3): **{'yes' if c['scoreContrastTested'] else 'no'}**",
          f"- RL beats imitation for indpen (H16-4 and its interval): **{'yes' if c['beatsImitation'] else 'no: seed-dependent'}**",
          f"- Selfish drift replicated (at least 3 of 4 individual runs with solo share above 0.5): "
          f"**{'yes' if c['driftReplicated'] else 'no'}** ({S['drift']} of 4)", '', '## Runs (means over the 15 main boards)', '',
          '| run | clues shared per held | solo share | welfare | gain over warm start |', '|---|---|---|---|---|']
    L += [f"| {n} | {_g(v['sharePerHeld'], 2)} | {_g(v['soloShare'], 2)} | {_g(v['welfare'])} | {_g(v['gain'])} |"
          for n, v in S['runs'].items()]
    L += ['', '## Seed spread (standard deviation over runs) and the between-arm difference', '',
          '| metric | individual | indpen | indpen - individual |', '|---|---|---|---|']
    L += [f"| {m} | {_g(v['individual'])} | {_g(v['indpen'])} | {_g(v['between'])} |" for m, v in S['spread'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def figRL16(out, S):
    "rl16.png: each run a point per arm and metric, the arm mean as a bar, the warm start dashed."
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, (m, title) in zip(axs, (('sharePerHeld', 'clues shared / clues held'), ('soloShare', 'solves made alone'),
                                   ('welfare', 'welfare per agent-turn'))):
        for y, (a, col) in enumerate((('individual', '#c0392b'), ('indpen', '#e67e22'))):
            v = [S['runs'][n][m] for n in RL16_ALL[a]]
            ax.barh(y, np.nanmean(v), color=col, alpha=0.25, height=0.6)
            ax.scatter(v, y + np.linspace(-0.18, 0.18, len(v)), color=col, zorder=3)
        ax.axvline(S['runs']['warm'][m], ls='--', color='k', lw=1)
        ax.set_yticks([0, 1], ['individual', 'loophole priced'])
        ax.set_title(title, fontsize=10)
    return _save(fig, out / 'rl16.png')


def syntheticRows(seed=0, share=(0.2, 0.9), solo=(0.7, 0.03), welfare=(0.1, 0.42), warm=0.27, seedSd=0.04, boardSd=0.08):
    "Evaluation rows (rows12, rows13, rows16) for the nine runs and the warm start: (individual, indpen) means per metric."
    rng, rows = np.random.default_rng(seed), dict(eval=[], eval_p13=[], new=[])
    where = {**{n: d for n, (d, _) in RL16_REUSED.items()}, **{n: 'new' for n in NEW}}
    model = {**{n: m for n, (_, m) in RL16_REUSED.items()}, **{n: n for n in NEW}}
    for n in ['warm'] + RL16_ALL['individual'] + RL16_ALL['indpen']:
        i = 0 if n.startswith('individual') else 1
        mu = dict(sharePerHeld=0.92, soloShare=0.0, welfare=warm) if n == 'warm' else \
            dict(sharePerHeld=share[i], soloShare=solo[i], welfare=welfare[i])
        off = {k: rng.normal(0, seedSd) for k in mu} if n != 'warm' else dict.fromkeys(mu, 0.0)
        for sd in MAIN_SEEDS:
            r = {k: float(np.clip(mu[k] + off[k] + rng.normal(0, boardSd), 0, None)) for k in mu}
            rows[where[n]].append(dict(level=f'{model[n]}|main', seed=sd, **r))
    return rows['eval'], rows['eval_p13'], rows['new']

## Tests

In [ ]:
import numpy as np
from explRL.rl16_report import *
from explRL.rl16_report import _mean

# AC-4: exact p-values
assert abs(permTest([5, 6, 7, 8, 9], [1, 2, 3, 4])['p'] - 1 / 126) < 1e-12
assert permTest([1, 1, 1, 1, 1], [1, 1, 1, 1])['p'] == 1.0
assert abs(signFlip([1, 2, 3, 4, 5])['p'] - 1 / 32) < 1e-12 and signFlip([0, 0, 0])['p'] == 1.0
assert permTest([1, 2, 3, 4, 5], [6, 7, 8, 9])['p'] == 1.0

In [ ]:
# AC-4: hierarchical bootstrap coverage on simulated data (200 simulations x 500 draws); undercovers with few runs
rng, cover = np.random.default_rng(1), 0
for s in range(200):
    g = {a: {f'{a}{k}': {b: v for b, v in enumerate(mu + rng.normal(0, 0.1) + rng.normal(0, 0.2, 15))} for k in range(n)}
         for a, mu, n in (('a', 0.3, 5), ('b', 0.1, 4))}
    lo, hi = hierBoot(g, lambda d: d['a'].mean() - d['b'].mean(), 500, seed=s)
    cover += lo <= 0.2 <= hi
assert 0.85 <= cover / 200 <= 0.99, cover / 200
print('coverage', cover / 200)

In [ ]:
# summaries on synthetic runs with a large effect: everything supported, verdicts follow the claim rules
r12, r13, r16 = syntheticRows()
S = rl16Summaries(r12, r13, r16, nboot=300)
assert all(S['hyp'][k]['supported'] for k in ('H16_1', 'H16_2', 'H16_3', 'H16_4'))
assert all(abs(S['hyp'][k]['p'] - 1 / 126) < 1e-12 for k in ('H16_1', 'H16_2', 'H16_3'))      # complete separation
assert all(abs(S['hyp'][k]['pHolm'] - 3 / 126) < 1e-12 for k in ('H16_1', 'H16_2', 'H16_3'))   # Holm over three equal p
assert S['hyp']['H16_4']['pHolm'] != S['hyp']['H16_4']['pHolm'] and abs(S['hyp']['H16_4']['p'] - 1 / 32) < 1e-12
assert S['claims'] == dict(sharingTested=True, scoreContrastTested=True, beatsImitation=True, driftReplicated=True)
assert S['drift'] == 4 and len(S['runs']) == 10
# no effect: nothing supported, the drift does not replicate, imitation not beaten
r12, r13, r16 = syntheticRows(seed=3, share=(0.9, 0.9), solo=(0.03, 0.03), welfare=(0.27, 0.27))
S0 = rl16Summaries(r12, r13, r16, nboot=300)
assert not any(S0['claims'].values()) and S0['drift'] == 0
# a board with no solves (soloShare nan) is skipped
r16[0] = dict(r16[0], soloShare=float('nan'))
S1 = rl16Summaries(r12, r13, r16, nboot=100)
n0 = r16[0]['level'].split('|')[0]
assert S1['runs'][n0]['soloShare'] == _mean([r['soloShare'] for r in r16 if r['level'] == f'{n0}|main'])
# AC-5: incomplete rows are refused
try:
    rl16Summaries(r12, r13, r16[1:], nboot=10)
    raise AssertionError('incomplete rows must be refused')
except ValueError:
    pass
# report and figure
import tempfile, pathlib
with tempfile.TemporaryDirectory() as d:
    p = writeRL16(pathlib.Path(d) / 'report.md', S, dict(mode='test', planHash='x', commit='y'))
    assert 'H16-1' in p.read_text() and figRL16(pathlib.Path(d), S).exists()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()